<a href="https://colab.research.google.com/github/aboettcher-sig/morocco-forests-sig/blob/main/scripts/utility/load_gee_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Load GEE exports to SQLite

Builds the analysis database from the Drive CSV exports produced by
pull_gee_data (run morroco_forest_exploration_2026_08_11_1300), including
Dynamic World.

The database is built on local disk and copied to Drive at the end, because
SQLite over the Drive mount is slow for the index-heavy work here.

Stages:
1. Copy the export CSVs from Drive to local disk.
2. Per dataset: recover the identity the export flattened away. `system:index`
   is `{image_id}_{row_counter}`; `.geo` is GeoJSON `[lon, lat]`. Produce a
   real `image_id`, a parsed `date`, and `lon`/`lat` columns.
3. Concatenate each dataset's files and deduplicate once across the whole
   group. The export used overlapping year windows, so the same scene appears
   in more than one file; per-file dedup would miss those.
4. Build `{table}_unique_locs` per dataset and assign `unique_loc_id`. Exact
   float equality is valid because every timestamp of a dataset samples the
   same fixed pixel grid.
5. Build the persistent read index `(unique_loc_id, date)` on every table.
   This is the structure the time-series reads run on: it covers both "all
   rows at a location" and "that location's series in date order".
6. Verify, then copy the finished database to Drive.

The database is rebuilt from scratch on every run, so deduplication happens
once in-flight and there are no post-hoc `_unique` shadow tables.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os

run_name = "morroco_forest_exploration_2026_08_18_1600"

# GEE Drive table exports land in a folder named after `folder=` at Drive ROOT.
drive_export_path = f"/content/drive/MyDrive/{run_name}"

# Local staging and database (built locally, copied to Drive at the end)
local_csv_path = f"/content/{run_name}"
db_path = f"/content/{run_name}.sqlite"

# Destination for the finished database on Drive
drive_db_path = f"/content/drive/MyDrive/morroco_forests/{run_name}/{run_name}.sqlite"

In [5]:
import shutil

os.makedirs(local_csv_path, exist_ok=True)

copied = 0
for item in os.listdir(drive_export_path):
    if item.endswith(".csv"):
        shutil.copy(os.path.join(drive_export_path, item), os.path.join(local_csv_path, item))
        copied += 1

print(f"Copied {copied} CSV files to {local_csv_path}")

Copied 319 CSV files to /content/morroco_forest_exploration_2026_08_18_1600


# Loader functions

In [3]:
import sqlite3
import polars as pl
import orjson
import pandas as pd


def sanitize_column_names(df):
    """SQLite-safe column names: replace ':', '.', ' ' with '_'."""
    df.columns = [col.replace(':', '_').replace('.', '_').replace(' ', '_') for col in df.columns]
    return df


def get_table_group(filename):
    """Map an export filename to its database table.

    Keys are tested in order; 'mTPI' precedes any broader terrain match so
    SRTM_mTPI_* lands in terrain_mTPI rather than a generic SRTM table.
    """
    groups = {
        "mTPI": "terrain_mTPI",
        "aspect": "terrain_aspect",
        "elevation": "terrain_elevation",
        "slope": "terrain_slope",
        "TerraClimate": "terraclimate",
        "CFSv2": "cfsv2",
        "DynamicWorld": "dynamic_world",
        "Landsat9": "landsat_9",
        "Landsat8": "landsat_8",
        "Landsat7": "landsat_7",
        "Landsat5": "landsat_5",
        "Sentinel2": "sentinel_2",
        "Sentinel1": "sentinel_1",
    }
    for key, value in groups.items():
        if key in filename:
            return value
    return None


def parse_date(image_id, table_group):
    """Parse an ISO date from image_id, per dataset naming convention.

    - landsat_*    : 'LC08_201035_20150410'                    -> third token, YYYYMMDD
    - terraclimate : '198501'                                   -> YYYYMM, first of month
    - cfsv2        : '2018030100' (YYYYMMDDHH, 6-hourly)         -> leading YYYYMMDD
    - dynamic_world: '20150627T110656_20150627T111031_T29SQV'  -> first token, YYYYMMDD prefix
    - sentinel_2   : same S2 granule id as dynamic_world         -> first token, YYYYMMDD prefix
    - sentinel_1   : 'S1A_IW_GRDH_1SDV_20191202T163455_...'      -> fifth token, YYYYMMDD prefix
    - terrain_*    : static, no date (image_id is '')
    """
    if not image_id:
        return None
    try:
        if table_group.startswith("landsat"):
            t = image_id.split("_")[2]
            return f"{t[0:4]}-{t[4:6]}-{t[6:8]}"
        if table_group == "terraclimate":
            return f"{image_id[0:4]}-{image_id[4:6]}-01"
        if table_group == "cfsv2":
            return f"{image_id[0:4]}-{image_id[4:6]}-{image_id[6:8]}"
        if table_group in ("dynamic_world", "sentinel_2"):
            t = image_id.split("_")[0]
            return f"{t[0:4]}-{t[4:6]}-{t[6:8]}"
        if table_group == "sentinel_1":
            t = image_id.split("_")[4]
            return f"{t[0:4]}-{t[4:6]}-{t[6:8]}"
    except (IndexError, ValueError):
        return None
    return None


def process_csv(file_path, table_group):
    """Read one export CSV and return a cleaned pandas DataFrame, or None.

    - image_id: `system:index` minus the trailing per-row counter that
      flatten() appends. Single-image exports (terrain) index rows as bare
      integers, so image_id is '' there.
    - date: parsed from image_id per dataset (None for terrain).
    - lon/lat: from `.geo`; GeoJSON coordinate order is [lon, lat].
    - within-file duplicates dropped (cross-file dedup happens after concat).
    """
    if os.stat(file_path).st_size == 0:
        print(f"Skipping empty file: {os.path.basename(file_path)}")
        return None

    try:
        df = pl.read_csv(file_path, infer_schema_length=1000)
    except pl.exceptions.NoDataError:
        print(f"Skipping empty file (NoDataError): {os.path.basename(file_path)}")
        return None

    if df.height == 0:
        print(f"Skipping header-only file: {os.path.basename(file_path)}")
        return None

    # image_id: strip the trailing `_{row_counter}` from system:index. Single-image
    # exports index rows as bare integers, which polars may read as Int, so cast
    # to str first; a bare counter has no image identity and yields ''.
    df = df.with_columns(pl.col("system:index").cast(pl.Utf8))
    df = df.with_columns(
        pl.col("system:index")
        .map_elements(lambda s: s.rsplit("_", 1)[0] if "_" in s else "", return_dtype=pl.Utf8)
        .alias("image_id")
    ).drop("system:index")

    # date per dataset convention
    df = df.with_columns(
        pl.col("image_id")
        .map_elements(lambda s: parse_date(s, table_group), return_dtype=pl.Utf8)
        .alias("date")
    )

    # lon/lat from .geo; GeoJSON coordinate order is [lon, lat]
    df = df.with_columns(
        pl.col(".geo").map_elements(
            lambda x: orjson.loads(x)["coordinates"] if x else [None, None],
            return_dtype=pl.List(pl.Float64)
        ).alias("coordinates")
    )
    df = df.with_columns(
        pl.col("coordinates").list.get(0).alias("lon"),
        pl.col("coordinates").list.get(1).alias("lat"),
    ).drop("coordinates", ".geo")

    df = df.unique()  # within-file duplicates
    df = df.to_pandas()
    df = sanitize_column_names(df)
    return df


def load_csv_files_to_db(folder_path, db_path):
    """Resumably build the database, one table per dataset, without piling rows in RAM.

    Files are streamed into their table one at a time and appended, so memory
    stays bounded by a single file rather than a whole dataset. Cross-file
    duplicates (overlapping year windows put the same scene in more than one
    file) are removed per dataset in SQL after its files are in.

    Resumable: each finished dataset is recorded in `_completed_groups` and
    skipped on a rerun, and the database is never wiped, so a crash is picked up
    where it left off. On the first run any table already present (from a prior
    non-resumable load) is treated as complete.
    """
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("PRAGMA temp_store=FILE;")  # keep the DISTINCT temp b-tree off the heap

    def table_exists(name):
        cur.execute("SELECT 1 FROM sqlite_master WHERE type='table' AND name=?", (name,))
        return cur.fetchone() is not None

    first_run = not table_exists("_completed_groups")
    cur.execute("CREATE TABLE IF NOT EXISTS _completed_groups (table_group TEXT PRIMARY KEY)")

    if first_run:
        # tables already present were written whole by a previous load; keep them
        cur.execute(
            "SELECT name FROM sqlite_master WHERE type='table' "
            "AND name NOT LIKE '\\_%' ESCAPE '\\' AND name NOT LIKE '%_unique_locs'"
        )
        for (name,) in cur.fetchall():
            cur.execute("INSERT OR IGNORE INTO _completed_groups VALUES (?)", (name,))
    conn.commit()

    done = {r[0] for r in cur.execute("SELECT table_group FROM _completed_groups")}

    grouped_files = {}
    for filename in sorted(os.listdir(folder_path)):
        if filename.endswith(".csv"):
            group = get_table_group(filename)
            if group:
                grouped_files.setdefault(group, []).append(os.path.join(folder_path, filename))
            else:
                print(f"No table group for: {filename}")

    for group, files in grouped_files.items():
        if group in done:
            print(f"{group}: already complete, skipping")
            continue

        # a partial table from an earlier interrupted run is rebuilt from scratch
        cur.execute(f"DROP TABLE IF EXISTS `{group}`")
        conn.commit()

        rows = 0
        for f in files:
            df = process_csv(f, group)
            if df is None or df.empty:
                continue
            # widen the table if this file introduces new band columns
            if table_exists(group):
                have = {r[1] for r in cur.execute(f"PRAGMA table_info(`{group}`)")}
                added = [c for c in df.columns if c not in have]
                for c in added:
                    cur.execute(f"ALTER TABLE `{group}` ADD COLUMN `{c}`")
                if added:
                    conn.commit()
            df.to_sql(group, conn, if_exists="append", index=False)
            rows += len(df)
            print(f"{group}: +{len(df)} rows from {os.path.basename(f)}")

        if not table_exists(group):
            print(f"{group}: no data in {len(files)} file(s), skipping")
            continue

        # cross-file dedup in SQL (on disk), then finalize and record completion
        cur.execute(f"CREATE TABLE `{group}__dedup` AS SELECT DISTINCT * FROM `{group}`")
        cur.execute(f"DROP TABLE `{group}`")
        cur.execute(f"ALTER TABLE `{group}__dedup` RENAME TO `{group}`")
        cur.execute(f"SELECT COUNT(*) FROM `{group}`")
        deduped = cur.fetchone()[0]
        cur.execute("INSERT OR IGNORE INTO _completed_groups VALUES (?)", (group,))
        conn.commit()
        print(f"{group}: {rows} rows -> {deduped} after cross-file dedup, done")

    conn.close()
    print("Load complete.")

# Build the database

In [ ]:
load_csv_files_to_db(local_csv_path, db_path)

cfsv2: already complete, skipping
dynamic_world: already complete, skipping
landsat_5: already complete, skipping
landsat_7: already complete, skipping
landsat_8: already complete, skipping
landsat_9: already complete, skipping
terrain_mTPI: already complete, skipping
Skipping empty file (NoDataError): Sentinel1_morroco_forest_exploration_2026_08_18_1600_1984_1984.csv
Skipping empty file (NoDataError): Sentinel1_morroco_forest_exploration_2026_08_18_1600_1984_1985.csv
Skipping empty file (NoDataError): Sentinel1_morroco_forest_exploration_2026_08_18_1600_1985_1986.csv
Skipping empty file (NoDataError): Sentinel1_morroco_forest_exploration_2026_08_18_1600_1986_1987.csv
Skipping empty file (NoDataError): Sentinel1_morroco_forest_exploration_2026_08_18_1600_1987_1988.csv
Skipping empty file (NoDataError): Sentinel1_morroco_forest_exploration_2026_08_18_1600_1988_1989.csv
Skipping empty file (NoDataError): Sentinel1_morroco_forest_exploration_2026_08_18_1600_1989_1990.csv
Skipping empty fi

In [ ]:
# Tables and row counts
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
for table in [t[0] for t in cursor.fetchall()]:
    cursor.execute(f"SELECT COUNT(*) FROM `{table}`;")
    print(f"{table}: {cursor.fetchone()[0]} rows")
conn.close()

In [ ]:
# Preview one row per data table
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
for table in [t[0] for t in cursor.fetchall()]:
    df = pd.read_sql_query(f"SELECT * FROM `{table}` LIMIT 1;", conn)
    print(f"--- {table} ---")
    print(df.iloc[0])
conn.close()

# Unique locations

Within one dataset every timestamp samples the same fixed pixel grid, so
repeated visits to a location have bit-identical lon/lat and exact float
equality is a valid join. The `(lat, lon)` index on each `_unique_locs`
table is what the keying join below looks into, and it stays for
coordinate-to-location lookup at query time.

In [ ]:
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE '\\_%' ESCAPE '\\' AND name NOT LIKE '%_unique_locs';")
data_tables = [t[0] for t in cursor.fetchall()]

for table in data_tables:
    cursor.execute(f"DROP TABLE IF EXISTS `{table}_unique_locs`;")
    cursor.execute(f"""
        CREATE TABLE `{table}_unique_locs` (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            lat FLOAT,
            lon FLOAT
        );
    """)
    cursor.execute(f"""
        INSERT INTO `{table}_unique_locs` (lat, lon)
        SELECT DISTINCT lat, lon FROM `{table}`
        WHERE lat IS NOT NULL AND lon IS NOT NULL;
    """)
    cursor.execute(f"CREATE INDEX IF NOT EXISTS idx_{table}_unique_locs_latlon ON `{table}_unique_locs` (lat, lon);")
    cursor.execute(f"SELECT COUNT(*) FROM `{table}_unique_locs`;")
    print(f"{table}: {cursor.fetchone()[0]} unique locations")

conn.commit()
conn.close()

# Assign unique_loc_id

Each data row gets the id of its location. The correlated lookup reads into
`{table}_unique_locs`, which is indexed on `(lat, lon)`.

In [ ]:
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

for table in data_tables:
    u = f"{table}_unique_locs"
    cursor.execute(f"ALTER TABLE `{table}` ADD COLUMN unique_loc_id INTEGER;")
    cursor.execute(f"""
        UPDATE `{table}`
        SET unique_loc_id = (
            SELECT id FROM `{u}`
            WHERE `{u}`.lat = `{table}`.lat AND `{u}`.lon = `{table}`.lon
        );
    """)
    print(f"{table}: keyed by unique_loc_id")

conn.commit()
conn.close()

# Read index for time-series retrieval

The transformer feed reads one location's series at a time:
`WHERE unique_loc_id = ? ORDER BY date`. The composite `(unique_loc_id, date)`
index serves that directly. Its leading column also covers plain
`WHERE unique_loc_id = ?` lookups (terrain tables, where date is null). This
is the persistent structure the reads run on, built last over the final state.

In [ ]:
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

for table in data_tables:
    cursor.execute(f"CREATE INDEX IF NOT EXISTS idx_{table}_loc_date ON `{table}` (unique_loc_id, date);")
    print(f"{table}: indexed on (unique_loc_id, date)")

conn.commit()
conn.close()

# Verification

In [ ]:
# Every row in every data table should carry a unique_loc_id
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
for table in data_tables:
    cursor.execute(f"SELECT COUNT(*) FROM `{table}` WHERE unique_loc_id IS NULL;")
    nulls = cursor.fetchone()[0]
    print(f"{table}: {'OK' if nulls == 0 else f'PROBLEM: {nulls} rows unkeyed'}")
conn.close()

In [ ]:
# Per-location series lengths for a time-series table (should equal that
# location's number of observed dates)
conn = sqlite3.connect(db_path)
check_table = "dynamic_world"
if check_table in data_tables:
    df = pd.read_sql_query(f"""
        SELECT unique_loc_id, COUNT(*) AS n_obs, COUNT(DISTINCT date) AS n_dates
        FROM `{check_table}`
        GROUP BY unique_loc_id
        ORDER BY unique_loc_id
        LIMIT 10;
    """, conn)
    print(df)
conn.close()

# Copy the database to Drive

In [ ]:
os.makedirs(os.path.dirname(drive_db_path), exist_ok=True)
shutil.copy(db_path, drive_db_path)
print(f"Database copied to {drive_db_path}")